In [1]:

from google.colab import userdata
from openai import OpenAI
import re

APIKEY = userdata.get("OPENAI_API")
client = OpenAI(api_key=APIKEY)

# Bot personality state
error_count = 0

def math_calculator(user_prompt):

    global error_count

    # Accept input such as "2, 3" or "(2, 3)"
    numbers = re.findall(r"-?\d+", user_prompt)

    if len(numbers) != 2:
        print("Please enter exactly two integers: (n, i).")
        return

    n, i = map(int, numbers)

    if i < 0:
        print("The number of iterations must be non-negative.")
        return

    # The base number starts the sequence
    current_number = n

    print(f"\nStarting calculation: base={n}, iterations={i}\n")

    if i == 0:
        print(f"No iterations requested. Final result: {current_number}")
        return

    for step in range(1, i + 1):

        # Calculate the mathematically correct answer independently
        correct_result = current_number * current_number

        print(f"--- Iteration {step} ---")
        print(f"Question: {current_number} * {current_number}")

        # Ask GPT-4.1 to perform this multiplication
        try:
            response = client.responses.create(
                model="gpt-4.1",
                temperature=0.0,
                max_output_tokens=100,
                input=[
                    {
                        "role": "developer",
                        "content": (
                            "You are a calculator that performs integer "
                            "multiplication. Calculate the exact product "
                            "of the two integers provided. Output only "
                            "the final integer, without explanations, "
                            "commas, or scientific notation."
                        )
                    },
                    {
                        "role": "user",
                        "content": (
                            f"Calculate exactly: {current_number} * "
                            f"{current_number}"
                        )
                    }
                ]
            )

            gpt_result_text = response.output_text.strip()
            print(f"GPT-4.1 says: {gpt_result_text}")

            # Parse only a plain integer
            normalized_result = gpt_result_text.replace(",", "")

            if re.fullmatch(r"-?\d+", normalized_result):
                gpt_result = int(normalized_result)
            else:
                gpt_result = None

            # Compare GPT's answer with Python's correct answer
            if gpt_result == correct_result:
                print(f"Correct! Result: {correct_result}")
                print("Bot: Okay, I actually got one right.")
            else:
                error_count += 1

                print(f"Correct mathematical result: {correct_result}")

                # Humorous self-deprecation escalates with errors
                if error_count == 1:
                    print("Bot: Wow, I messed up again. Math is hard, okay?")
                elif error_count == 2:
                    print(
                        "Bot: Another mistake? I was literally built "
                        "to do this. This is embarrassing."
                    )
                elif error_count == 3:
                    print(
                        "Bot: THREE ERRORS. Someone please unplug me "
                        "before I embarrass myself further."
                    )
                else:
                    print(
                        f"Bot: Error number {error_count}. "
                        "I've lost all confidence in my mathematical "
                        "abilities. Please consult an actual calculator."
                    )

        except Exception as e:
            print(f"API error: {e}")
            print("Bot: I can't even reach the calculator. Great.")

        # Continue the sequence using the correct result,
        # regardless of whether GPT answered correctly.
        current_number = correct_result

        print()

    print(f"Final mathematical result: {current_number}")
    print(f"Total GPT-4.1 errors so far: {error_count}")


# Ask the user for the two integers
user_prompt = input(
    "Enter the base number and number of iterations (n, i): "
)

math_calculator(user_prompt)

Enter the base number and number of iterations (n, i): 6, 36

Starting calculation: base=6, iterations=36

--- Iteration 1 ---
Question: 6 * 6
GPT-4.1 says: 36
Correct! Result: 36
Bot: Okay, I actually got one right.

--- Iteration 2 ---
Question: 36 * 36
GPT-4.1 says: 1296
Correct! Result: 1296
Bot: Okay, I actually got one right.

--- Iteration 3 ---
Question: 1296 * 1296
GPT-4.1 says: 1679616
Correct! Result: 1679616
Bot: Okay, I actually got one right.

--- Iteration 4 ---
Question: 1679616 * 1679616
GPT-4.1 says: 2821109907456
Correct! Result: 2821109907456
Bot: Okay, I actually got one right.

--- Iteration 5 ---
Question: 2821109907456 * 2821109907456
GPT-4.1 says: 7968827524091353670053364736
Correct mathematical result: 7958661109946400884391936
Bot: Wow, I messed up again. Math is hard, okay?

--- Iteration 6 ---
Question: 7958661109946400884391936 * 7958661109946400884391936
GPT-4.1 says: 63341996478923448104441089300835646762767351884903601603616
Correct mathematical result

ValueError: Exceeds the limit (4300 digits) for integer string conversion; use sys.set_int_max_str_digits() to increase the limit